## Classification Dataset 

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Edge-case dataset
X, y = make_classification(
    n_samples=2000,
    n_features=15,
    n_informative=5,      # truly useful
    n_redundant=5,        # highly correlated → edge case for max_features
    n_repeated=0,
    n_classes=2,
    weights=[0.7, 0.3],   # mild imbalance
    flip_y=0.03,          # 3% label noise
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("Dataset shape:", X.shape)
print("Informative + redundant + noise features → perfect for testing RF mechanisms")

Dataset shape: (2000, 15)
Informative + redundant + noise features → perfect for testing RF mechanisms


## Sklearn Model 

In [2]:
from sklearn.ensemble import RandomForestClassifier

rf_sklearn = RandomForestClassifier(
    n_estimators=200,           # more trees = smoother
    max_depth=None,             # let trees grow full
    min_samples_split=2,
    max_features='sqrt',        # critical for decorrelation on redundant features
    bootstrap=True,
    oob_score=True,             # we will compare with our OOB
    n_jobs=-1,
    random_state=42,
    criterion='gini',
    class_weight='balanced'     # handles mild imbalance
)



In [3]:
rf_sklearn.fit(X_train, y_train)
print("Sklearn OOB score:", rf_sklearn.oob_score_)


Sklearn OOB score: 0.91


In [4]:
y_preds_sklearn = rf_sklearn.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_preds_sklearn))
print("Top 5 features (should down-weight redundant & noise):", 
      np.argsort(rf_sklearn.feature_importances_)[-5:])

Test accuracy: 0.9083333333333333
Top 5 features (should down-weight redundant & noise): [ 5 12  4  9  7]


## Random Forest From Scratch 


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import mode
from collections import defaultdict

class RandomForestClassifierScratch:
    def __init__(
        self,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        max_features='sqrt',
        bootstrap=True,
        oob_score=False,
        random_state=None,
        n_jobs=1,
        criterion='gini',
        **tree_kwargs
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features          
        self.bootstrap = bootstrap
        self.oob_score = oob_score
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.criterion = criterion
        self.tree_kwargs = tree_kwargs
        self.trees = []
        self.oob_score_ = None
        self.feature_importances_ = None

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        n_samples = X.shape[0]
        rng = np.random.RandomState(self.random_state)

        self.trees = []
        oob_predictions = [[] for _ in range(n_samples)]   # list of predictions per sample

        for i in range(self.n_estimators):
            # ====================== BOOTSTRAP SAMPLING ====================== #
            if self.bootstrap:
                sample_idx = rng.choice(n_samples, size=n_samples, replace=True)
                # OOB mask (samples never selected)
                in_bag = np.zeros(n_samples, dtype=bool)
                in_bag[sample_idx] = True
                oob_idx = np.where(~in_bag)[0]
            else:
                sample_idx = np.arange(n_samples)
                oob_idx = np.array([], dtype=int)

            # ====================== BUILD TREE WITH ALL PARAMETERS ====================== #
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features,
                criterion=self.criterion,
                random_state=rng.randint(2**32) if self.random_state is not None else None,
                **self.tree_kwargs
            )
            tree.fit(X[sample_idx], y[sample_idx])
            self.trees.append(tree)

            # ====================== OOB PREDICTIONS =================================== #
            if self.oob_score and len(oob_idx) > 0:
                oob_pred = tree.predict(X[oob_idx])
                for j, idx in enumerate(oob_idx):
                    oob_predictions[idx].append(oob_pred[j])
        
        # ================== Compute OOB Score ========================================== #
        